[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C24_Inference_Serving_Course/01_paged_attention/01_paged_attention.ipynb)

# 01 · PagedAttention 与 KV 管理（用 numpy 做模拟器+账本）

目标：把 **KV cache 碎片化**、**分页（block + block table）**、**按需分配/回收**、**copy-on-write 前缀共享**、**碎片率/利用率账本** 用 numpy 实现出来，并用 `assert` 钉死核心不变量。

路线：KV cache 大小账本 → 连续分配的碎片 → 分页 KV 管理器 → 块表间接寻址（对拍连续存储） → CoW 引用计数 → 利用率对比 → ✏️ 练习 → 📖 答案 → 🧪 真实模型配置胶囊。

> 心智模型：**显存 = 一个「块→占用者/引用计数」的整数账本；block table = 逻辑块→物理块的 list；CoW = 写前查引用计数**。我们写的是*机制与正确性*，不是吞吐。

## 1 · KV cache 大小：为什么它是显存命门

KV cache 字节 ≈ `2 × 层数 × KV头数 × head_dim × 序列长 × batch × 字节/元素`。
先把这个账本写出来，建立「KV 随序列、并发线性膨胀」的量感。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def kv_bytes(layers, kv_heads, head_dim, seq, batch, dbytes=2):
    return 2 * layers * kv_heads * head_dim * seq * batch * dbytes

cfg = dict(layers=32, kv_heads=8, head_dim=128, dbytes=2)   # Llama-3-8B 量级, fp16
per_tok = kv_bytes(seq=1, batch=1, **cfg)
print(f'每 token KV ≈ {per_tok/1024:.1f} KB')
for seq, bs in [(2048,1),(8192,1),(8192,32)]:
    print(f'  seq={seq:5d} batch={bs:3d} -> {kv_bytes(seq=seq,batch=bs,**cfg)/1e9:5.2f} GB')
assert kv_bytes(seq=2000,batch=1,**cfg) == 2*kv_bytes(seq=1000,batch=1,**cfg)
assert kv_bytes(seq=1000,batch=4,**cfg) == 4*kv_bytes(seq=1000,batch=1,**cfg)
print('✅ KV 随 seq、batch 线性增长 —— 8B 模型也能被 KV 撑爆显存')

## 2 · 连续分配的碎片：利用率为何只有 20–40%

传统方案为每个请求按 **max_len** 预留一整段连续显存，但实际生成长度远小于 max_len → 大量**内部碎片**。
用「token 槽」为单位（忽略层/头的常数因子）做账本：给定显存槽数、max_len、一批请求的真实长度，算利用率。

In [ ]:
def contiguous_alloc(total_slots, max_len, true_lengths):
    '''连续分配：每个请求占 max_len 槽。返回 (容纳的请求数, 利用率).'''
    slots_per_req = max_len
    capacity = total_slots // slots_per_req       # 能放下几个请求
    admitted = true_lengths[:capacity]
    used = sum(admitted)                          # 真正有效的槽
    allocated = len(admitted) * slots_per_req     # 预留(占用)的槽
    util = used / allocated if allocated else 0.0
    return len(admitted), util, used, allocated

total_slots = 100
max_len = 20
true_lengths = [3, 7, 2, 5, 6, 4, 8, 3]            # 真实生成长度，都远小于 max_len
n, util, used, alloc = contiguous_alloc(total_slots, max_len, true_lengths)
print(f'连续分配: 容纳 {n} 个请求, 用了 {used}/{alloc} 槽, 利用率 {util:.0%}')
assert n == 5, '100//20 = 5 个请求就占满'
assert util < 0.4, '按 max_len 预留 -> 利用率必然很低'
print('✅ 连续分配利用率 < 40%：预留的 max_len 绝大部分是内部碎片')

## 3 · 分页 KV 管理器：按需取、用完还

把 KV 切成固定大小的 **block**，用一个 **free list** 管理空闲物理块，每个请求一张 **block table**（逻辑块→物理块）。
下面从零写一个分页管理器，支持：为请求按需分配块、追加 token（写满才取新块）、释放请求（块还回 free list）。

In [ ]:
class PagedKVManager:
    def __init__(self, n_blocks, block_size):
        self.B = block_size
        self.free = list(range(n_blocks))          # 空闲物理块列表
        self.n_blocks = n_blocks
        self.block_table = {}                      # req_id -> [物理块,...]
        self.length = {}                           # req_id -> 已用 token 数

    def _need_blocks(self, n_tokens):
        return (n_tokens + self.B - 1) // self.B    # ceil

    def allocate(self, req_id, n_tokens):
        '''prefill：一次为 n_tokens 个 token 分配足够的块。'''
        need = self._need_blocks(n_tokens)
        assert need <= len(self.free), 'OOM: 空闲块不足'
        blocks = [self.free.pop() for _ in range(need)]
        self.block_table[req_id] = blocks
        self.length[req_id] = n_tokens
        return blocks

    def append_token(self, req_id):
        '''decode：追加 1 个 token；当前块写满才取新块（按需，不预留）。'''
        L = self.length[req_id]
        if L % self.B == 0:                         # 现有块正好写满
            assert self.free, 'OOM: 无法追加'
            self.block_table[req_id].append(self.free.pop())
        self.length[req_id] = L + 1

    def free_request(self, req_id):
        '''请求结束：所有块还回 free list，立即可复用。'''
        for pb in self.block_table.pop(req_id):
            self.free.append(pb)
        self.length.pop(req_id)

    def used_blocks(self):
        return self.n_blocks - len(self.free)

mgr = PagedKVManager(n_blocks=25, block_size=4)        # 100 槽
mgr.allocate('A', 3)                                   # 3 token -> 1 块
mgr.allocate('B', 7)                                   # 7 token -> 2 块
print('分配 A(3),B(7) 后用了', mgr.used_blocks(), '块')
assert mgr.used_blocks() == 1 + 2
for _ in range(5): mgr.append_token('A')              # A 再生成 5 个 -> 共 8 token -> 2 块
assert len(mgr.block_table['A']) == 2
mgr.free_request('B')                                  # B 结束，2 块还回
assert len(mgr.free) == 25 - 2                         # 只剩 A 的 2 块在用
print('✅ 分页管理器：按需取块、写满才扩、结束即还 —— 这就是 vLLM BlockManager 的骨架')

## 4 · 块表间接寻址：物理乱序，逻辑正确（核心不变量）

KV 物理上分散，注意力靠 `phys = block_table[pos // B]`、`off = pos % B` 间接寻址。
**核心不变量**：无论物理块怎么乱序存放，按块表读回的 KV 必须**逐位等于**连续存储。我们存真实的小 KV 向量来验证。

In [ ]:
def scatter_store(kv, pool, block_table, B):
    '''把连续的 kv[seq,dim] 按 block_table 写进物理块池 pool[n_blocks,B,dim].'''
    seq = kv.shape[0]
    for pos in range(seq):
        lb, off = pos // B, pos % B
        pb = block_table[lb]
        pool[pb, off] = kv[pos]

def gather_load(pool, block_table, seq, B):
    '''按 block_table 间接寻址读回逻辑顺序的 kv.'''
    dim = pool.shape[-1]
    out = np.zeros((seq, dim))
    for pos in range(seq):
        lb, off = pos // B, pos % B
        pb = block_table[lb]
        out[pos] = pool[pb, off]
    return out

seq, dim, B = 10, 4, 4
kv_true = rng.standard_normal((seq, dim))              # 参考：连续 KV
n_blocks = (seq + B - 1) // B
block_table = list(rng.permutation(8)[:n_blocks])      # 故意挑乱序的物理块
pool = np.zeros((8, B, dim))
scatter_store(kv_true, pool, block_table, B)
kv_read = gather_load(pool, block_table, seq, B)
print('block_table (逻辑->物理):', block_table)
assert np.array_equal(kv_read, kv_true), '块表读回必须逐位等于连续 KV！'
print('✅ 物理块乱序，按块表读回逐位等于连续存储 —— PagedAttention 的正确性基石')

## 5 · Copy-on-Write：零拷贝前缀共享 + 写时分叉

多个序列共享相同前缀块（refcount>1），零拷贝省显存；谁要往共享块写就**复制出私有块**再写，互不干扰。
给每块加 **引用计数**：共享+1，写入时若 count>1 触发 CoW，释放时减到 0 才回收。

In [ ]:
class CoWBlockPool:
    def __init__(self, n_blocks):
        self.free = list(range(n_blocks))
        self.refcount = [0] * n_blocks
        self.copies = 0                            # 统计发生了多少次 CoW

    def alloc(self):
        pb = self.free.pop(); self.refcount[pb] = 1; return pb

    def share(self, pb):
        '''另一个序列共享已有块：引用计数+1，零拷贝。'''
        self.refcount[pb] += 1; return pb

    def write(self, pb):
        '''要往 pb 写入：独占(count==1)直接写；共享(count>1)则 CoW。
           返回真正应写入的块号(可能是新块).'''
        if self.refcount[pb] == 1:
            return pb                               # 独占，原地写
        new = self.free.pop()                       # CoW：分配私有块
        self.refcount[new] = 1
        self.refcount[pb] -= 1                      # 我不再用旧块
        self.copies += 1
        return new                                  # 调用方把块表改指 new 并拷贝内容

    def release(self, pb):
        self.refcount[pb] -= 1
        if self.refcount[pb] == 0:
            self.free.append(pb)

pool = CoWBlockPool(n_blocks=10)
p = pool.alloc()                                   # 候选A 分配前缀块
pool.share(p)                                      # 候选B 共享同一前缀块（零拷贝）
assert pool.refcount[p] == 2 and pool.copies == 0
print(f'前缀块 refcount={pool.refcount[p]}, 拷贝次数={pool.copies}  <- 共享零拷贝')
wb = pool.write(p)                                 # A 要写前缀块 -> 触发 CoW
assert wb != p and pool.copies == 1 and pool.refcount[p] == 1
print(f'A 写入触发 CoW -> 新块 {wb}; 原块 refcount 降回 {pool.refcount[p]} (B 仍安全共享)')
pool.release(p)                                    # B 结束
assert pool.refcount[p] == 0 and p in pool.free
print('✅ CoW：共享不拷贝、写时才分叉、引用计数归零才回收 —— 前缀/并行采样省显存的底座')

## 6 · 把账算清：分页 vs 连续的利用率与并发

用同一批请求喂给两种分配器，对比 **KV 利用率** 与 **能容纳的并发数**——亲眼看到 PagedAttention 的吞吐红利从哪来。

In [ ]:
def paged_alloc(total_slots, block_size, true_lengths):
    '''分页：每请求占 ceil(len/B) 块。尽量多接请求直到块用完。'''
    B = block_size
    free_blocks = total_slots // B
    admitted, used_slots, alloc_slots = [], 0, 0
    for L in true_lengths:
        need = (L + B - 1) // B
        if need <= free_blocks:
            free_blocks -= need
            admitted.append(L)
            used_slots += L
            alloc_slots += need * B
    util = used_slots / alloc_slots if alloc_slots else 0.0
    return len(admitted), util

total_slots, max_len, B = 100, 20, 4
lengths = [3,7,2,5,6,4,8,3,5,2,6,4]
nc, uc, *_ = contiguous_alloc(total_slots, max_len, lengths)
np_, up = paged_alloc(total_slots, B, lengths)
print(f'连续分配 : 容纳 {nc:2d} 请求, 利用率 {uc:.0%}')
print(f'分页分配 : 容纳 {np_:2d} 请求, 利用率 {up:.0%}')
assert np_ > nc, '分页应容纳更多请求'
assert up > uc + 0.3, '分页利用率应远高于连续'
print(f'✅ 同样 {total_slots} 槽显存，分页多接 {np_-nc} 个请求、利用率高 {(up-uc):.0%} —— 这就是 2-4x 吞吐红利的来源')

---
## ✏️ 练习 1：块分配器（OOM 检测）

实现 `BlockAllocator`：`__init__(n_blocks)` 建空闲列表；`alloc(k)` 取 k 块返回列表（不足则 `raise MemoryError`）；`free(blocks)` 还回；`num_free()` 返回剩余。要求**还回的块能被再次分配**（不泄漏、不重复）。

In [ ]:
class BlockAllocator:
    def __init__(self, n_blocks):
        # TODO: 用一个 list 维护空闲物理块号 [0..n_blocks-1]
        #       提示：成员变量别命名为 self.free（会和下面的 free() 方法重名）
        raise NotImplementedError
    def alloc(self, k):
        # TODO: 取 k 个空闲块返回；不足则 raise MemoryError
        raise NotImplementedError
    def free(self, blocks):
        # TODO: 把 blocks 里的块还回空闲列表
        raise NotImplementedError
    def num_free(self):
        raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
al = BlockAllocator(8)
assert al.num_free() == 8
b = al.alloc(5); assert len(b) == 5 and al.num_free() == 3
al.free(b[:2]); assert al.num_free() == 5          # 还回 2 块
raised = False
try:
    al.alloc(100)
except MemoryError:
    raised = True
assert raised, 'alloc 超量应 raise MemoryError'
# 还回的块能再次分配
b2 = al.alloc(5); assert len(b2) == 5
assert len(set(b2)) == 5, '分配的块不应重复'
print('✅ 练习 1 通过：分配/回收/OOM 检测正确，块可复用不泄漏')

## ✏️ 练习 2：block table 查找

实现 `logical_to_physical(block_table, pos, B)`：给逻辑位置 `pos`，返回 `(物理块号, 块内偏移)`。
再实现 `read_kv(pool, block_table, pos, B)` 从物理块池读出该位置的 KV 向量。

In [ ]:
def logical_to_physical(block_table, pos, B):
    # TODO: 返回 (block_table[pos//B], pos % B)
    raise NotImplementedError

def read_kv(pool, block_table, pos, B):
    # TODO: 用 logical_to_physical 定位，返回 pool[pb, off]
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
B = 4
bt = [5, 2, 8]                                   # 逻辑块 0,1,2 -> 物理 5,2,8
assert logical_to_physical(bt, 0, B) == (5, 0)
assert logical_to_physical(bt, 6, B) == (2, 2)    # 6//4=1->物理2, 6%4=2
assert logical_to_physical(bt, 9, B) == (8, 1)
pool = rng.standard_normal((10, B, 3))
# 与直接索引对拍
for pos in [0, 6, 9]:
    pb, off = logical_to_physical(bt, pos, B)
    assert np.array_equal(read_kv(pool, bt, pos, B), pool[pb, off])
print('✅ 练习 2 通过：块表寻址与物理读取正确')

## ✏️ 练习 3：CoW 引用计数

实现 `cow_write(refcount, free, pb)`：模拟 CoW 写入。`refcount` 是列表、`free` 是空闲块栈、`pb` 是要写的块。
若 `refcount[pb]==1` 返回 `(pb, False)`（原地写、未拷贝）；若 `>1` 则从 `free` 取新块、新块 refcount=1、旧块 refcount-1、返回 `(新块, True)`。

In [ ]:
def cow_write(refcount, free, pb):
    # TODO: 按 refcount[pb] 是否为 1 决定原地写还是 CoW
    #       返回 (实际写入的块号, 是否发生了拷贝)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
refcount = [0]*6; free = [5,4,3]
refcount[0] = 1                                  # 块0 独占
out, copied = cow_write(refcount, free, 0)
assert out == 0 and copied is False, '独占应原地写'
refcount[1] = 2                                  # 块1 被共享
out, copied = cow_write(refcount, free, 1)
assert copied is True, '共享块写入应 CoW'
assert refcount[out] == 1 and refcount[1] == 1, '新块refcount=1, 旧块减到1'
assert out not in free, '取出的新块应离开 free'
print('✅ 练习 3 通过：CoW 在独占时零拷贝、共享时正确分叉')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
class BlockAllocator:
    def __init__(self, n_blocks):
        self._free = list(range(n_blocks))   # 注意：别叫 self.free，会和 free() 方法重名
    def alloc(self, k):
        if k > len(self._free):
            raise MemoryError(f'need {k}, have {len(self._free)}')
        return [self._free.pop() for _ in range(k)]
    def free(self, blocks):
        self._free.extend(blocks)
    def num_free(self):
        return len(self._free)

In [ ]:
# 练习 2 参考答案
def logical_to_physical(block_table, pos, B):
    return block_table[pos // B], pos % B

def read_kv(pool, block_table, pos, B):
    pb, off = logical_to_physical(block_table, pos, B)
    return pool[pb, off]

In [ ]:
# 练习 3 参考答案
def cow_write(refcount, free, pb):
    if refcount[pb] == 1:
        return pb, False
    new = free.pop()
    refcount[new] = 1
    refcount[pb] -= 1
    return new, True

---
## 🧪 真实数据胶囊：用真实模型配置算 KV 显存与最大并发

下面是几个**真实**开源模型的配置（来自其 HuggingFace `config.json`）。用它们算 KV cache 大小、以及给定显存能容纳多少并发——把分页的「省显存」接到真实数字。

> 用 GQA 的 `num_key_value_heads`（KV 头数，常远小于 query 头数）才是 KV cache 的正确口径。

In [ ]:
# 真实模型配置（layers, kv_heads(num_key_value_heads), head_dim）
MODELS = {
    'Llama-3-8B':   dict(layers=32, kv_heads=8,  head_dim=128),
    'Llama-3-70B':  dict(layers=80, kv_heads=8,  head_dim=128),
    'Mistral-7B':   dict(layers=32, kv_heads=8,  head_dim=128),
    'Qwen2-7B':     dict(layers=28, kv_heads=4,  head_dim=128),
}

def kv_per_token_bytes(m, dbytes=2):
    return 2 * m['layers'] * m['kv_heads'] * m['head_dim'] * dbytes

def max_concurrent(m, kv_budget_gb, seq_len, dbytes=2):
    per_req = kv_per_token_bytes(m, dbytes) * seq_len
    return int(kv_budget_gb * 1e9 // per_req)

print(f"{'model':14s}{'KB/token':>10s}{'并发@(2k,40GB KV)':>20s}")
for name, m in MODELS.items():
    kbt = kv_per_token_bytes(m) / 1024
    conc = max_concurrent(m, kv_budget_gb=40, seq_len=2048)
    print(f'{name:14s}{kbt:>9.1f}K{conc:>20d}')
# Llama-3-8B 每 token KV = 2*32*8*128*2 = 131072 字节 = 128 KB
assert kv_per_token_bytes(MODELS['Llama-3-8B']) == 2*32*8*128*2
print('\n观察：70B 与 8B 的 KV 头数都是 8（GQA 的功劳），所以单 token KV 只差在层数。')

**🧪 胶囊练习**：实现 `slots_to_blocks_saving(seq_len, block_size, max_len)`：对一个真实长度 `seq_len` 的请求，返回 `(连续按max_len占的槽, 分页占的槽, 省下的槽)`。用它说明长 max_len、短实际长度时分页省得最多。

In [ ]:
def slots_to_blocks_saving(seq_len, block_size, max_len):
    # TODO: 连续占 max_len 槽；分页占 ceil(seq_len/block_size)*block_size 槽；返回三元组
    raise NotImplementedError

In [ ]:
# 自测
cont, paged, saved = slots_to_blocks_saving(seq_len=10, block_size=16, max_len=2048)
assert cont == 2048
assert paged == 16             # ceil(10/16)=1 块 *16
assert saved == 2048 - 16
print(f'seq=10, max_len=2048, B=16 -> 连续占 {cont} 槽, 分页占 {paged} 槽, 省下 {saved} 槽')
print('✅ 胶囊练习通过：实际短、预留长时，分页几乎省下全部预留浪费')

In [ ]:
# 📖 胶囊参考答案
def slots_to_blocks_saving(seq_len, block_size, max_len):
    cont = max_len
    paged = ((seq_len + block_size - 1) // block_size) * block_size
    return cont, paged, cont - paged

---
### 小结
- **KV cache** 把 decode 从 O(n²) 算力降到 O(n) 读取，但显存随 seq×batch 线性膨胀，是服务的命门。
- 连续分配按 max_len 预留 → **内部/外部碎片**，利用率仅 20–40%。
- **PagedAttention** = OS 分页：KV 切块 + **block table** 间接寻址，逻辑连续物理分散，利用率 ~96%。
- 核心不变量：物理块乱序，按块表读回**逐位等于**连续 KV。
- **Copy-on-Write** + 引用计数：前缀/并行采样零拷贝共享，写时才分叉、计数归零才回收。

下一站：**模块 02 · Continuous Batching** —— 显存管好了，怎么让 batch 动态进出、把 GPU 一直喂满。